# Export Model ke ONNX -- CNNsa-BiLSTM & MViTv2

Notebook ini export kedua model (checkpoint hasil `Training/CNNsaBiLSTM/Train.ipynb` dan
`Training/MViTv2/TrainMViT.ipynb`) ke format ONNX, lalu validasi output-nya sama dengan
model PyTorch asli.

Prasyarat: kedua checkpoint (`cnnsa_bilstm_best.pt`, `mvit_v2_best.pt`) sudah ada di
`Training/Output/{cnnsa_bilstm,mvit_v2}/checkpoints/`.

> Catatan: tahap ini murni konversi format + validasi numerik di PC (CPU) -- BUKAN app
> Android. Cuma CNNsa-BiLSTM yang direncanakan masuk ke app Android nantinya (MViTv2 terlalu
> besar/lambat untuk mobile -- lihat `README.md` bagian Model Evaluation). Export MViTv2 ke ONNX
> tetap dilakukan untuk kelengkapan dokumentasi/pembanding, bukan untuk deployment.


## 1. Setup & Cek Checkpoint

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from Export.Config.export_config import create_output_dirs, verify_checkpoints_exist

create_output_dirs()
verify_checkpoints_exist()


## 2. Export CNNsa-BiLSTM ke ONNX

In [ ]:
from Export.Scripts.export_cnnsa_bilstm import export_cnnsa_bilstm

export_cnnsa_bilstm()


## 3. Export MViTv2 ke ONNX

> Model besar (34M parameter) -- proses tracing/export bisa makan waktu beberapa menit.

In [ ]:
from Export.Scripts.export_mvit_v2 import export_mvit_v2

export_mvit_v2()


## 4. Validasi: PyTorch vs ONNX Runtime

Bandingkan output kedua model (PyTorch asli vs hasil load ONNX lewat ONNX Runtime) memakai
sample yang SAMA -- kalau ada data test asli, pakai itu; kalau belum, random dummy input.
Validasi jalan di **CPU** (`CPUExecutionProvider`) -- alasan lengkap ada di docstring
`Export/Scripts/validate_onnx.py` dan `Export Setup.md`.

In [ ]:
from Export.Scripts.validate_onnx import validate_all

all_ok = validate_all()
print()
print("SEMUA VALID -- siap dipakai" if all_ok else "ADA YANG GAGAL VALIDASI -- cek log di atas")


## Output Akhir

```
Export/Output/onnx/
  cnnsa_bilstm.onnx   <- kandidat untuk masuk app Android
  mvit_v2.onnx        <- baseline pembanding riset, TIDAK untuk app Android
```

**Belum dilakukan** (langkah berikutnya, di luar notebook ini): integrasi `cnnsa_bilstm.onnx`
ke project Android baru (CameraX + MediaPipe Hands on-device + ONNX Runtime Mobile) -- belum
ada kodenya.